In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project root
PROJECT_ROOT = Path.cwd().parent

# Load cleaned data
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_survey.csv"

df = pd.read_csv(DATA_PATH)

print("=" * 60)
print("AWARENESS PREDICTION MODEL")
print("=" * 60)

print(f"Dataset shape: {df.shape}")

print("\nTarget distribution:")
print(df["outlook_awareness_1"].value_counts())

print("\nTarget percentage:")
print(
    df["outlook_awareness_1"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

AWARENESS PREDICTION MODEL
Dataset shape: (11109, 101)

Target distribution:
outlook_awareness_1
No     5623
Yes    5486
Name: count, dtype: int64

Target percentage:
outlook_awareness_1
No     50.62
Yes    49.38
Name: proportion, dtype: float64


In [2]:
# ---------------------------------------------------------
# STAGE 8A: SELECT MODEL FEATURES
# ---------------------------------------------------------

target = "outlook_awareness_1"

features = [
    "age_group",
    "gender",
    "education_level",
    "residence_type",
    "employment_status",
    "brand_discovery_method",
    "brand_awareness_driver",
    "digital_ad_platform",
    "digital_campaign_frequency",
    "digital_purchase_frequency",
    "digital_monthly_spend",
    "previous_purchase"
]

# Keep only required columns
model_df = df[features + [target]].copy()

print("=" * 60)
print("MODEL DATASET")
print("=" * 60)

print(f"Rows    : {model_df.shape[0]:,}")
print(f"Columns : {model_df.shape[1]}")

print("\nFeatures:")
for feature in features:
    print(f"- {feature}")

print(f"\nTarget:\n- {target}")

MODEL DATASET
Rows    : 11,109
Columns : 13

Features:
- age_group
- gender
- education_level
- residence_type
- employment_status
- brand_discovery_method
- brand_awareness_driver
- digital_ad_platform
- digital_campaign_frequency
- digital_purchase_frequency
- digital_monthly_spend
- previous_purchase

Target:
- outlook_awareness_1


In [3]:
# Check missing values before modeling

missing = model_df.isnull().sum()

print("Missing values:")
print(missing[missing > 0])

if missing.sum() == 0:
    print("\n✓ No missing values in modeling dataset.")
else:
    print("\n⚠ Missing values found. We will handle them before modeling.")

Missing values:
Series([], dtype: int64)

✓ No missing values in modeling dataset.


In [4]:
# ---------------------------------------------------------
# STAGE 8B: ENCODE TARGET
# ---------------------------------------------------------

model_df["awareness_target"] = (
    model_df["outlook_awareness_1"]
    .str.strip()
    .str.lower()
    .map({
        "no": 0,
        "yes": 1
    })
)

# Check whether every value was successfully mapped
print("Encoded target distribution:")
print(model_df["awareness_target"].value_counts(dropna=False))

print("\nTarget percentages:")
print(
    model_df["awareness_target"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

# Safety check
if model_df["awareness_target"].isna().sum() > 0:
    raise ValueError(
        "Some target values could not be encoded. "
        "Check the original outlook_awareness_1 values."
    )

print("\n✓ Target successfully encoded.")

Encoded target distribution:
awareness_target
0    5623
1    5486
Name: count, dtype: int64

Target percentages:
awareness_target
0    50.62
1    49.38
Name: proportion, dtype: float64

✓ Target successfully encoded.


In [5]:
# ---------------------------------------------------------
# STAGE 8C: TRAIN / TEST SPLIT
# ---------------------------------------------------------

from sklearn.model_selection import train_test_split

# X = predictor variables
X = model_df[features].copy()

# y = target variable
y = model_df["awareness_target"].copy()

# 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("=" * 60)
print("TRAIN / TEST SPLIT")
print("=" * 60)

print(f"Total observations : {len(X):,}")
print(f"Training set       : {len(X_train):,}")
print(f"Test set           : {len(X_test):,}")

print("\nTraining target distribution:")
print(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTest target distribution:")
print(
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

TRAIN / TEST SPLIT
Total observations : 11,109
Training set       : 8,887
Test set           : 2,222

Training target distribution:
awareness_target
0    50.61
1    49.39
Name: proportion, dtype: float64

Test target distribution:
awareness_target
0    50.63
1    49.37
Name: proportion, dtype: float64


In [6]:
# ---------------------------------------------------------
# STAGE 8D: IDENTIFY NUMERIC AND CATEGORICAL FEATURES
# ---------------------------------------------------------

numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numeric features:")
for col in numeric_features:
    print(f"- {col}")

print("\nCategorical features:")
for col in categorical_features:
    print(f"- {col}")

print("\nFeature count:")
print(f"Numeric      : {len(numeric_features)}")
print(f"Categorical  : {len(categorical_features)}")
print(f"Total        : {len(numeric_features) + len(categorical_features)}")

Numeric features:

Categorical features:
- age_group
- gender
- education_level
- residence_type
- employment_status
- brand_discovery_method
- brand_awareness_driver
- digital_ad_platform
- digital_campaign_frequency
- digital_purchase_frequency
- digital_monthly_spend
- previous_purchase

Feature count:
Numeric      : 0
Categorical  : 12
Total        : 12


C:\Users\DIVYANSHU KASHYAP\AppData\Local\Temp\ipykernel_26484\3658828772.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [7]:
# ---------------------------------------------------------
# STAGE 8D: CREATE PREPROCESSING PIPELINE
# ---------------------------------------------------------

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_features
        ),
        (
            "numeric",
            "passthrough",
            numeric_features
        )
    ]
)

print("✓ Preprocessing pipeline created.")

✓ Preprocessing pipeline created.


In [8]:
# ---------------------------------------------------------
# STAGE 8E: LOGISTIC REGRESSION
# ---------------------------------------------------------

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Create the complete modeling pipeline
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

# Train the model
logistic_model.fit(X_train, y_train)

print("✓ Logistic Regression model trained successfully.")


✓ Logistic Regression model trained successfully.


In [9]:
# ---------------------------------------------------------
# STAGE 8F: MODEL EVALUATION
# ---------------------------------------------------------

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Predictions
y_pred = logistic_model.predict(X_test)

# Probability of being aware
y_prob = logistic_model.predict_proba(X_test)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("=" * 60)
print("LOGISTIC REGRESSION PERFORMANCE")
print("=" * 60)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Not Aware", "Aware"]
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

LOGISTIC REGRESSION PERFORMANCE
Accuracy  : 0.5000
Precision : 0.4925
Recall    : 0.4184
F1 Score  : 0.4524
ROC-AUC   : 0.4976

Classification Report:
              precision    recall  f1-score   support

   Not Aware       0.51      0.58      0.54      1125
       Aware       0.49      0.42      0.45      1097

    accuracy                           0.50      2222
   macro avg       0.50      0.50      0.50      2222
weighted avg       0.50      0.50      0.50      2222


Confusion Matrix:
[[652 473]
 [638 459]]


In [10]:
# ---------------------------------------------------------
# STAGE 8G: SAVE MODEL PERFORMANCE
# ---------------------------------------------------------

from pathlib import Path

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model_results = pd.DataFrame({
    "model": ["Logistic Regression"],
    "accuracy": [accuracy],
    "precision": [precision],
    "recall": [recall],
    "f1_score": [f1],
    "roc_auc": [roc_auc]
})

results_path = OUTPUT_DIR / "awareness_model_results.csv"

model_results.to_csv(results_path, index=False)

print(f"✓ Model results saved to:")
print(results_path)

✓ Model results saved to:
c:\Users\DIVYANSHU KASHYAP\Desktop\OUTLOOK-CUSTOMER-INTELLIGENCE\outputs\tables\awareness_model_results.csv
